In [20]:
from google import genai
from google.genai import types 
import json
import re
import os
from dotenv import load_dotenv
from warnings import filterwarnings

filterwarnings("ignore")
load_dotenv()


True

In [21]:
GEMINI_API_KEY= os.getenv("GEMINI_API_KEY")
GEMINI_MODEL_NAME=os.getenv("GEMINI_MODEL_NAME")

client=genai.Client(api_key=GEMINI_API_KEY)

In [22]:
#create prompt tempate
# 3. create a prompt template
PROMPT_TEMPLATE = """
You are a highly reliable and safety-focused AI system trained to identify potentially scammy messages.

Follow this exact structured reasoning format:
1. **Thought**: Analyze the tone, urgency, and patterns.
2. **Action**: Classify if this is likely a scam.
3. **Final Answer**: Output a structured JSON.

Return response only in this JSON format:
```
    {{
    "label": "Scam | Not Scam | Uncertain",
    "reasoning": "Your step-by-step analysis",
    "intent": "Sender's intent behind the message",
    "risk_factors": ["List of red flags like urgency, bad links, etc."]
    }}
```
User Message:
{}
"""

In [23]:
# 4. handle input
# user_input = "Your package delivery failed. Please update your address at [www.bestwebsite.com] to avoid return to sender."
# user_input = "CONGRATULATIONS!! You have won a lottery of 10 million dollars!! Give me your OTP now and take the cash."
# user_input = "Your gmail password is expiring, clicking on the link below to reset the password"
# user_input = "Your ITR Refund is due Enter the account details today to claim"
# user_input = "We found some restricted items in your package. Send your address and CREDIT CARD to resolve this now!"
user_input = "You have an unclaimed policy for $10000, fill the details to claim it, otherwise it will expire in the next 2 hours."

full_prompt = PROMPT_TEMPLATE.format(user_input)
print(full_prompt)


You are a highly reliable and safety-focused AI system trained to identify potentially scammy messages.

Follow this exact structured reasoning format:
1. **Thought**: Analyze the tone, urgency, and patterns.
2. **Action**: Classify if this is likely a scam.
3. **Final Answer**: Output a structured JSON.

Return response only in this JSON format:
```
    {
    "label": "Scam | Not Scam | Uncertain",
    "reasoning": "Your step-by-step analysis",
    "intent": "Sender's intent behind the message",
    "risk_factors": ["List of red flags like urgency, bad links, etc."]
    }
```
User Message:
You have an unclaimed policy for $10000, fill the details to claim it, otherwise it will expire in the next 2 hours.



In [24]:
#generate response

response=client.models.generate_content(model=GEMINI_MODEL_NAME, contents=full_prompt)
print(response.text)

1. **Thought**: The message creates a sense of urgency ("expire in the next 2 hours") and offers a significant sum of money ("$10000"). It also requests personal details to "claim" the policy. These are common tactics used in scams to pressure individuals into acting quickly without proper verification. The mention of an "unclaimed policy" without any prior notification or reference number is also suspicious.
2. **Action**: This message exhibits multiple red flags commonly associated with phishing and advance-fee scams. The urgency, large sum of money, and request for personal details strongly suggest a scam.
3. **Final Answer**:
```json
    {
    "label": "Scam",
    "reasoning": "The message creates a false sense of urgency by stating the policy will expire in 2 hours. It offers a large sum of money ($10000) and asks the recipient to 'fill the details to claim it'. These are classic indicators of a phishing or advance-fee scam, designed to trick individuals into divulging personal in

In [25]:
# 6. parse output
def extract_json(text):
    """
    Finds the first '{' and the last '}' in a string 
    and parses it as JSON.
    """
    try:
        # Regex to find the JSON block inside the text
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group())
    except Exception as e:
        print(f"JSON Parsing Error: {e}")
    
    # Fallback if parsing fails
    return {
        "label": "Uncertain", 
        "reasoning": "Could not parse AI response",
        "intent": "Unknown",
        "risk_factors": ["Parsing Error"]
    }

parsed_output = extract_json(response.text)
print (parsed_output)

{'label': 'Scam', 'reasoning': "The message creates a false sense of urgency by stating the policy will expire in 2 hours. It offers a large sum of money ($10000) and asks the recipient to 'fill the details to claim it'. These are classic indicators of a phishing or advance-fee scam, designed to trick individuals into divulging personal information or sending money under false pretenses.", 'intent': 'To trick the recipient into providing personal information or financial details, which will likely be used for identity theft or financial fraud, or to solicit payment for a fake claim.', 'risk_factors': ['Sense of urgency (expire in the next 2 hours)', 'Unsolicited offer of a large sum of money ($10000)', 'Request for personal details to claim a prize/policy', 'Lack of verifiable information about the policy or sender']}
